# RQ1: Badanie Stabilności Wyjaśnień XAI (Analiza Wielobudżetowa)
Ocena odporności metod atrybucji na perturbacje promptu w funkcji budżetu obliczeniowego (IG, LIME, SHAP, Occlusion, Grad×Input).

### 1. Sprawdzenie akceleratora GPU

In [ ]:
!nvidia-smi
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB) | bf16: {torch.cuda.is_bf16_supported()}")

### 2. Dysk Google, repozytorium i instalacja zależności

In [ ]:
from google.colab import drive
import os

# 1. Montowanie Dysku Google
drive.mount('/content/drive')
DRIVE_RESULTS = "/content/drive/MyDrive/llm_xai_results"
os.makedirs(DRIVE_RESULTS, exist_ok=True)

# 2. Konfiguracja GitHub (dla repozytorium prywatnego podaj token PAT lub dodaj GH_TOKEN w Secrets)
GITHUB_TOKEN = "" #@param {type:"string"}
REPO_OWNER_REPO = "jakubstachyra/llm-xai" #@param {type:"string"}
BRANCH = "main" #@param {type:"string"}

try:
    from google.colab import userdata
    token_from_secrets = userdata.get('GH_TOKEN')
    if token_from_secrets:
        GITHUB_TOKEN = token_from_secrets
except Exception:
    pass

if GITHUB_TOKEN.strip():
    CLONE_URL = f"https://{GITHUB_TOKEN.strip()}@github.com/{REPO_OWNER_REPO}.git"
else:
    CLONE_URL = f"https://github.com/{REPO_OWNER_REPO}.git"

REPO_DIR = "/content/llm-xai"

if not os.path.exists(REPO_DIR):
    res = !git clone -b {BRANCH} {CLONE_URL} {REPO_DIR}
    print("\n".join(res))
    if not os.path.exists(REPO_DIR):
        raise RuntimeError("Błąd klonowania! Jeśli repozytorium jest prywatne, wklej Personal Access Token (PAT) w pole GITHUB_TOKEN lub dodaj jako sekret GH_TOKEN.")
    %cd {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git checkout {BRANCH}
    !git pull origin {BRANCH}

# 3. Pakiety (uaktualniamy numpy i pakiety zależne, aby uniknąć błędu StringDType w transformers/phi)
!pip install -q --upgrade numpy "transformers>=4.43.0,<4.47.0" "accelerate>=0.30.0" "datasets>=2.19.0" "shap>=0.46.0" scikit-learn scipy pandas matplotlib huggingface_hub

# 4. Token HuggingFace (dla modeli gated)
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
except Exception:
    pass

### 3. Szybki test potoku (Smoke test: 2 próbki)

In [ ]:
!python scripts/run_rq1_grid.py \
    --model microsoft/phi-2 \
    --samples 2 \
    --methods occlusion,grad_x_input,ig,lime,shap \
    --ig-grid 5 \
    --lime-grid 50 \
    --shap-grid 50 \
    --output-dir /content/smoke_test \
    --no-resume

### 4. Główny eksperyment RQ1 (Pełna siatka parametrów)

In [ ]:
model = "microsoft/phi-2" #@param ["microsoft/phi-2", "meta-llama/Llama-3.1-8B", "Qwen/Qwen2.5-7B"]
samples = 50 #@param {type:"integer"}
seed = 42 #@param {type:"integer"}
methods = "occlusion,grad_x_input,ig,lime,shap" #@param {type:"string"}

ig_grid = "10,20,50,100,200,300" #@param {type:"string"}
lime_grid = "50,150,500,1000,2500,5000" #@param {type:"string"}
shap_grid = "100,250,500,1000,2500,5000" #@param {type:"string"}

max_run_samples = 0 #@param {type:"integer"}
no_noise_floor = False #@param {type:"boolean"}

tag = model.split("/")[-1]
output_dir = f"{DRIVE_RESULTS}/rq1_grid_{tag}"
limit_flag = f"--max-run-samples {max_run_samples}" if max_run_samples > 0 else ""
noise_flag = "--no-noise-floor" if no_noise_floor else ""

!python scripts/run_rq1_grid.py \
    --model {model} \
    --samples {samples} \
    --seed {seed} \
    --methods {methods} \
    --ig-grid {ig_grid} \
    --lime-grid {lime_grid} \
    --shap-grid {shap_grid} \
    {limit_flag} \
    {noise_flag} \
    --output-dir {output_dir}

### 5. Tabela podsumowująca wyniki

In [ ]:
import os
import pandas as pd
from IPython.display import display

summary_csv = os.path.join(output_dir, "rq1_grid_summary.csv")
if os.path.exists(summary_csv):
    df_summary = pd.read_csv(summary_csv)
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 1000)
    display(df_summary.round(3))
else:
    print(f"Plik {summary_csv} nie istnieje.")

### 6. Wykres stabilność vs koszt (Krzywa Pareto)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

summary_csv = os.path.join(output_dir, "rq1_grid_summary.csv")
if os.path.exists(summary_csv):
    df = pd.read_csv(summary_csv)
    plt.figure(figsize=(10, 6), dpi=150)
    colors = {"Occlusion": "#1f77b4", "GradientXInput": "#d62728", "IntegratedGradients": "#2ca02c", "LIME": "#9467bd", "SHAP": "#ff7f0e"}
    
    for family, group in df.groupby("method_family"):
        group = group.sort_values("forward_passes_mean")
        plt.plot(group["forward_passes_mean"], group["spearman_mean"], marker="o", label=family, color=colors.get(family, "black"), linewidth=2)
        for _, row in group.iterrows():
            if pd.notna(row["param_value"]):
                plt.annotate(f"{int(row['param_value'])}", (row["forward_passes_mean"], row["spearman_mean"]), textcoords="offset points", xytext=(0, 7), ha="center", fontsize=8)
                
    plt.xscale("log")
    plt.xlabel("Średnia liczba operacji Forward Pass (skala log)")
    plt.ylabel(r"Stabilność (Spearman $\rho$)")
    plt.title(f"RQ1: Stabilność vs Koszt Obliczeniowy – {tag}", fontweight="bold")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend(title="Metoda")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "rq1_pareto_curve.png"))
    plt.show()
else:
    print("Brak pliku podsumowania.")